# Fine-tuning DistilBERT on AG News — the transformer half of `news-topic-classifier`

This notebook fine-tunes `distilbert-base-uncased` for 4-way news topic classification
(World / Sports / Business / Sci/Tech) on the real AG News benchmark, and evaluates it on
the **exact same held-out test set** already used to measure this project's classical
baseline (TF-IDF + logistic regression) — so the comparison at the end is apples-to-apples,
not two different numbers on two different splits.

**Why this runs here instead of in the project's own codebase:** the sandbox this project
was built in can't reach `huggingface.co` to download pretrained weights (confirmed directly
— `httpx.get(...)` to the Hub returns a hard `403 Forbidden` at the network-proxy level, not
a library or auth issue). Colab has full internet access and a free GPU, so this notebook is
the actual fine-tuning run; the baseline in the repo itself is real, measured, and already
verified without needing this notebook at all.

**Before running:** `Runtime → Change runtime type → T4 GPU` (not required, but a CPU
runtime will take much longer — expect ~15-25 min on GPU vs. 2-3+ hours on CPU for 3 epochs
over 120,000 examples).

**After running:** download the zipped model this notebook produces at the end, extract it
into `news-topic-classifier/models/ag_news_distilbert/` in your local copy of the repo, and
the FastAPI service will pick it up automatically and show it side by side with the baseline.


## 1. Install dependencies

Colab already has PyTorch; this adds the Hugging Face stack.

`torchvision` is uninstalled on purpose: installing these packages pulls in a slightly newer
`torch` as a side dependency, which can end up mismatched with the `torchvision` Colab
preinstalled. `datasets` has optional video-loading support that probes `torchvision` even
though this notebook is pure text and never touches it -- with a mismatched version that probe
crashes with `ImportError: cannot import name 'VideoReader' from 'torchvision.io'` the first
time `trainer.train()` runs. Since nothing here needs `torchvision`, removing it sidesteps the
crash entirely rather than fighting version pins.


In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn
!pip uninstall -y torchvision -q


## 2. Load the real AG News data

Pulled from the same plain-CSV mirror the project's own `scripts/fetch_data.py` uses
(`mhjabreel/CharCnn_Keras` on GitHub — a well-known, widely-used mirror of the canonical
Zhang/Zhao/LeCun 2015 AG News release), rather than `datasets.load_dataset("ag_news")`,
specifically so the **test split is byte-for-byte identical** to the one the baseline was
scored on — 120,000 training rows, 7,600 test rows, 4 balanced classes.


In [ ]:
import pandas as pd

TRAIN_URL = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"
TEST_URL = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv"
COLUMN_NAMES = ["class_id", "title", "description"]

train_df = pd.read_csv(TRAIN_URL, header=None, names=COLUMN_NAMES)
test_df = pd.read_csv(TEST_URL, header=None, names=COLUMN_NAMES)

print(f"train: {len(train_df)} rows, test: {len(test_df)} rows")
train_df.head()


In [ ]:
CLASS_NAMES = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
# HF expects 0-indexed labels; AG News' own CSVs are 1-indexed (1=World..4=Sci/Tech).
id2label = {i: CLASS_NAMES[i + 1] for i in range(4)}
label2id = {v: k for k, v in id2label.items()}

def prep(df):
    df = df.copy()
    df["text"] = df["title"].fillna("") + ". " + df["description"].fillna("")
    df["label"] = df["class_id"] - 1
    return df[["text", "label"]]

train_prepped = prep(train_df)
test_prepped = prep(test_df)
print(id2label)


## 3. Tokenize and build HF datasets

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256, padding="max_length")

train_ds = Dataset.from_pandas(train_prepped, preserve_index=False).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_prepped, preserve_index=False).map(tokenize, batched=True)

train_ds = train_ds.remove_columns(["text"])
test_ds = test_ds.remove_columns(["text"])
train_ds.set_format("torch")
test_ds.set_format("torch")
print(train_ds)


## 4. Load the pretrained model and fine-tune

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=4, id2label=id2label, label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

training_args = TrainingArguments(
    output_dir="./ag_news_distilbert_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=200,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()


## 5. Evaluate on the held-out test set, and compare against the real baseline

The baseline numbers below are hardcoded from the actual, already-measured run of
`app/baseline.py` in the repo (TF-IDF + logistic regression, 40,000 training rows, evaluated
on this same 7,600-row test set) — not a placeholder, the real result. This cell tells you,
with your own fine-tuned model's real numbers, whether the transformer actually earned its
extra complexity and compute cost, or not.


In [ ]:
from sklearn.metrics import f1_score as sk_f1

BASELINE_ACCURACY = 0.9046052631578947
BASELINE_MACRO_F1 = 0.9044454320450393
BASELINE_PER_CLASS_F1 = {
    "World": 0.9094,
    "Sports": 0.9612,
    "Business": 0.8690,
    "Sci/Tech": 0.8782,
}

eval_results = trainer.evaluate()
predictions = trainer.predict(test_ds)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

per_class = sk_f1(labels, preds, average=None)
transformer_per_class_f1 = {id2label[i]: float(f) for i, f in enumerate(per_class)}

print("=" * 60)
print(f"{'Metric':<14}{'Baseline':>12}{'DistilBERT':>14}{'Delta':>12}")
print("=" * 60)
acc_delta = eval_results["eval_accuracy"] - BASELINE_ACCURACY
f1_delta = eval_results["eval_macro_f1"] - BASELINE_MACRO_F1
print(f"{'Accuracy':<14}{BASELINE_ACCURACY:>12.4f}{eval_results['eval_accuracy']:>14.4f}{acc_delta:>+12.4f}")
print(f"{'Macro F1':<14}{BASELINE_MACRO_F1:>12.4f}{eval_results['eval_macro_f1']:>14.4f}{f1_delta:>+12.4f}")
print()
for cls in id2label.values():
    b = BASELINE_PER_CLASS_F1[cls]
    t = transformer_per_class_f1[cls]
    print(f"  {cls:<10} baseline={b:.4f}  distilbert={t:.4f}  delta={t - b:+.4f}")
print("=" * 60)
if acc_delta > 0:
    print(f"DistilBERT beat the baseline by {acc_delta*100:.2f} percentage points of accuracy -- measured, not assumed.")
else:
    print(f"DistilBERT did NOT beat the baseline on accuracy in this run ({acc_delta*100:.2f}pp) -- report this honestly, don't hide it.")


## 6. Save the fine-tuned model and download it

Copy the extracted folder into `news-topic-classifier/models/ag_news_distilbert/` in your
local repo afterward -- the FastAPI service loads from that exact path.


In [ ]:
import json
import shutil

OUT_DIR = "ag_news_distilbert"
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

results_summary = {
    "baseline": {
        "accuracy": BASELINE_ACCURACY,
        "macro_f1": BASELINE_MACRO_F1,
        "per_class_f1": BASELINE_PER_CLASS_F1,
    },
    "distilbert_finetuned": {
        "accuracy": float(eval_results["eval_accuracy"]),
        "macro_f1": float(eval_results["eval_macro_f1"]),
        "per_class_f1": transformer_per_class_f1,
    },
}
with open(f"{OUT_DIR}/results_summary.json", "w") as f:
    json.dump(results_summary, f, indent=2)
print(json.dumps(results_summary, indent=2))

shutil.make_archive("ag_news_distilbert", "zip", OUT_DIR)
print("\nZipped to ag_news_distilbert.zip -- download it below.")


In [ ]:
from google.colab import files
files.download("ag_news_distilbert.zip")


## 7. Next step

1. Unzip `ag_news_distilbert.zip` into `news-topic-classifier/models/ag_news_distilbert/`
   in your local repo (so `config.json`, `model.safetensors`, `tokenizer.json`, etc. sit
   directly in that folder — not inside another nested subfolder).
2. Run the service locally: `uvicorn app.main:app --reload`, open `http://localhost:8000/`,
   and the landing page will show the transformer's predictions next to the baseline's for
   any headline you type in, instead of the "not loaded yet" message.
3. `results_summary.json` (saved alongside the model) has the exact numbers from this run —
   worth sending back so the README's comparison table reflects your real result rather than
   a placeholder.
